# Forecast model estimation and training

This notebook retains the supplied econometric and machine-learning analysis. The implemented cost target is log USD per ton, despite the legacy name ln_cost_tonkm. The cost ANN reports in-sample metrics; the trade ANN uses a random split within each year/product group. These are not interchangeable with a future-year holdout. Read the methodological findings in AUDIT.md before interpreting results.

**Status:** publication review candidate; a complete clean-kernel execution has not been verified. Saved outputs and execution counts have been cleared. See README.md for inputs and run order.


## Workspace and configuration


In [ ]:
from pathlib import Path
import os

# Launch Jupyter in the research-notebooks directory (or one of its subdirectories).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "publication_manifest.json").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research-notebooks folder.")
WORK = ROOT / "work" / "03_forecast_model_training"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("Working directory:", WORK)
# Required input files belong here; outputs are written here too.
# work/ is excluded from the publication package and from Git by default.


In [ ]:
_database_engine = None

def get_database_engine():
    global _database_engine
    if _database_engine is None:
        from sqlalchemy import create_engine
        database_url = os.environ.get("THESIS_DATABASE_URL")
        if not database_url:
            raise RuntimeError("Set THESIS_DATABASE_URL outside the notebook; do not save credentials in cells.")
        _database_engine = create_engine(database_url)
    return _database_engine


In [ ]:
import re
def read_research_table(table_name):
    """Read an exact table snapshot from work/<stage>/tables, or an explicitly configured database."""
    import pandas as pd
    path = WORK / "tables" / (table_name + ".parquet")
    if path.exists():
        return pd.read_parquet(path)
    if not os.environ.get("THESIS_DATABASE_URL"):
        raise FileNotFoundError(f"Supply {path} or configure THESIS_DATABASE_URL.")
    # Names used below are fixed identifiers in the supplied research code.
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", table_name):
        raise ValueError("Invalid table identifier")
    return pd.read_sql(f'SELECT * FROM "{table_name}"', get_database_engine())


In [ ]:
# Pre processing
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# STEP 0: Connect to PostgreSQL

df_billateral_trade_combined_data = read_research_table("billateral_trade_combined_data")
df_bltr_multivar = df_billateral_trade_combined_data[~df_billateral_trade_combined_data['production_unit'].isna()]
df_bltr_multivar = df_bltr_multivar.drop_duplicates(subset=['year','product_label', 'origin_label','destination_label'], keep='first')
df_bltr_multivar['tons_value_allmodes'] = df_bltr_multivar['kilograms_value_allmodes']*0.001
df_bltr_multivar['tons_value_seamodes'] = df_bltr_multivar['kilograms_value_seamodes']*0.001
df_bltr_multivar['maritime_trade_share'] = (df_bltr_multivar['tons_value_seamodes']/df_bltr_multivar['tons_value_allmodes']).fillna(0)

In [ ]:
import numpy as np

epsilon = 1e-6  # tiny positive value to avoid log(0)

df_bltr_multivar['transport_work_km_seamodes'] = (
    df_bltr_multivar['transport_work_in_tonkm_value_seamodes'] /
    df_bltr_multivar['tons_value_seamodes'].replace(0, np.nan)
).fillna(0)
df_bltr_multivar['transport_expend_per_ton_seamodes'] = (
    df_bltr_multivar['transport_expenditure_us_value_seamodes'] /
    df_bltr_multivar['tons_value_seamodes'].replace(0, np.nan)
).fillna(0)
# Derived calorific supply
df_bltr_multivar["Origin_Daily_Calorific_Food"] = (
    df_bltr_multivar["origin_daily_calorific_food_per_capita"] * df_bltr_multivar["origin_population_total"]
)
df_bltr_multivar["Destination_Daily_Calorific_Food"] = (
    df_bltr_multivar["destination_daily_calorific_food_per_capita"] * df_bltr_multivar["destination_population_total"]
)

# Log transformations with epsilon intercept
epsilon = 1e-6

df_bltr_multivar["Log_Sea_Trade_Volume_Tons"] = np.log(
    np.where(df_bltr_multivar["tons_value_seamodes"] > epsilon,
             df_bltr_multivar["tons_value_seamodes"],
             epsilon)
)
df_bltr_multivar["Log_All_Trade_Volume_Tons"] = np.log(
    np.where(df_bltr_multivar["tons_value_allmodes"] > epsilon,
             df_bltr_multivar["tons_value_allmodes"],
             epsilon)
)
df_bltr_multivar["Log_Transport_Cost"] = np.log(
    np.where(df_bltr_multivar["seamode_transport_cost_intensity_in_us_per_tonkm_value"] > epsilon,
             df_bltr_multivar["seamode_transport_cost_intensity_in_us_per_tonkm_value"],
             epsilon)
)
'seamode_transport_cost_intensity_in_us_per_tonkm_value','origin_gdp_usd_current','destination_gdp_usd_current','origin_production_qty','destination_daily_calorific_food_per_capita','destination_production_qty','Origin_Daily_Calorific_Food'
df_bltr_multivar["Log_Origin_GDP"] = np.log(
    np.where(df_bltr_multivar["origin_gdp_usd_current"] > epsilon,
             df_bltr_multivar["origin_gdp_usd_current"],
             epsilon)
)

df_bltr_multivar["Log_Destination_GDP"] = np.log(
    np.where(df_bltr_multivar["destination_gdp_usd_current"] > epsilon,
             df_bltr_multivar["destination_gdp_usd_current"],
             epsilon)
)

df_bltr_multivar["Origin_Daily_Calorific_Food"] = (
    df_bltr_multivar["origin_daily_calorific_food_per_capita"] * df_bltr_multivar["origin_population_total"]
)
df_bltr_multivar["Destination_Daily_Calorific_Food"] = (
    df_bltr_multivar["destination_daily_calorific_food_per_capita"] * df_bltr_multivar["destination_population_total"]
)

df_bltr_multivar["Log_Origin_Daily_Calorific_Food"] = np.log(
    np.where(df_bltr_multivar["Origin_Daily_Calorific_Food"] > epsilon,
             df_bltr_multivar["Origin_Daily_Calorific_Food"],
             epsilon)
)
df_bltr_multivar["Log_Destination_Daily_Calorific_Food"] = np.log(
    np.where(df_bltr_multivar["Destination_Daily_Calorific_Food"] > epsilon,
             df_bltr_multivar["Destination_Daily_Calorific_Food"],
             epsilon)
)
df_bltr_multivar['Log_origin_production_qty'] = np.log(
    np.where(df_bltr_multivar['origin_production_qty'] > epsilon,
             df_bltr_multivar['origin_production_qty'],
             epsilon)
)
df_bltr_multivar['Log_destination_production_qty'] = np.log(
    np.where(df_bltr_multivar['destination_production_qty'] > epsilon,
             df_bltr_multivar['destination_production_qty'],
             epsilon)
)
df_bltr_multivar['Log_sea_transport_work'] = np.log(
    np.where(df_bltr_multivar['transport_work_km_seamodes'] > epsilon,
             df_bltr_multivar['transport_work_km_seamodes'],
             epsilon)
)

df_bltr_multivar['Log_cost_per_ton_seamodes'] = np.log(
    np.where(df_bltr_multivar['transport_expend_per_ton_seamodes'] > epsilon,
             df_bltr_multivar['transport_expend_per_ton_seamodes'],
             epsilon)
)

**Publication review note.** Original preprocessing is retained: policy flags are forward-filled across the current row order without grouping by country or corridor. Review before changing or interpreting these imputations.


In [ ]:
#Computed data
for col in ["tons_value_seamodes", "tons_value_allmodes"]:
    df_bltr_multivar[col] = df_bltr_multivar[col].fillna(0)
# Example: safe imputation rule
for col in ["cepii_contig", "cepii_comlang_ethno","lsbci"]:
    df_bltr_multivar[col] = df_bltr_multivar[col].fillna(0)

# Policy variables: conservative filling
for col in ["cepii_wto_o", "cepii_wto_d",'cepii_eu_o','cepii_eu_d', "cepii_fta_wto"]:
    df_bltr_multivar[col] = df_bltr_multivar[col].fillna(method="ffill").fillna(0)

In [ ]:
df_bltr_multivar["cepii_dist_original"] = df_bltr_multivar["cepii_dist"]

# Domestic pairs
df_bltr_multivar.loc[df_bltr_multivar["origin_isocode"] == df_bltr_multivar["destination_isocode"], "cepii_dist"] = 1.0

# Regional mean fill
df_bltr_multivar["cepii_dist"] = df_bltr_multivar.groupby(["origin_region", "destination_region"])["cepii_dist"].transform(
    lambda x: x.fillna(x.mean())
)

# Final fallback: global median
global_median = df_bltr_multivar["cepii_dist"].median()
df_bltr_multivar["cepii_dist"] = df_bltr_multivar["cepii_dist"].fillna(global_median)

# Add imputation flag
df_bltr_multivar["dist_imputed_flag"] = (df_bltr_multivar["cepii_dist_original"].isna()).astype(int)

# Log-transform safely
df_bltr_multivar["Log_Distance"] = np.log(df_bltr_multivar["cepii_dist"])

In [ ]:
predictors_all = [
    # Stage 1A
    "Log_Origin_GDP", "Log_Destination_GDP",
    "Log_Distance", "cepii_contig", "cepii_comlang_ethno",
    "cepii_wto_o", "cepii_wto_d", "cepii_fta_wto",
    # Stage 1B extensions
    "Log_origin_production_qty", "Log_destination_production_qty",
    "Log_Origin_Daily_Calorific_Food", "Log_Destination_Daily_Calorific_Food",
    # Stage 2C
    "Log_cost_per_ton_seamodes", "Log_sea_transport_work", "lsbci",
]

missing_predictors = df_bltr_multivar[predictors_all].isna().sum()
print("\n=== Missing Predictors Summary ===")
print(missing_predictors[missing_predictors > 0])

# Forecast Modelling: Transport Cost

In [ ]:
# Pre processing
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# STEP 0: Connect to PostgreSQL

df_fbx_global = read_research_table("extract_fbx_global_cleaned")
df_bdi_yearly = read_research_table("balticdryindex_data_yearly")
df_bdi_yearly['bdi_avg_usdprice'] = df_bdi_yearly['Price']
df_bltr_multivar = df_bltr_multivar.merge(
        df_fbx_global[['year','approx_avg_fb_rate_usd']],
        on=['year'],
        how="left"
    )
df_bltr_multivar = df_bltr_multivar.merge(
        df_bdi_yearly[['year','bdi_avg_usdprice']],
        on=['year'],
        how="left"
    )
df_bltr_multivar['log_avg_fb_rate_usd'] = np.log(
    np.where(df_bltr_multivar['approx_avg_fb_rate_usd'] > epsilon,
             df_bltr_multivar['approx_avg_fb_rate_usd'],
             epsilon)
)

**Publication review note.** The legacy cost_tonkm_sea / ln_cost_tonkm names actually use transport expenditure divided by tons under the supplied defaults. The Gamma GLM uses the level target with a log link. Numerical model code is retained.


In [ ]:
import numpy as np
import pandas as pd
import patsy
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ======================================================
# 0) Small helpers
# ======================================================
def _zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

def _safe_log(x, eps=1e-9):
    return np.log(np.clip(x, eps, None))

def _vif_table(X_df):
    try:
        vifs = [variance_inflation_factor(X_df.values, i) for i in range(X_df.shape[1])]
        return pd.DataFrame({"feature": X_df.columns, "VIF": vifs})
    except Exception:
        return pd.DataFrame({"feature": [], "VIF": []})


# ======================================================
# 1) Feature Engineering
# ======================================================
def build_features(
    df: pd.DataFrame,
    *,
    col_cost_tonkm="transport_expend_per_ton_seamodes",
    col_exp_usd="transport_expenditure_us_value_seamodes",
    col_work_tonkm="transport_work_in_tonkm_value_seamodes",
    col_tons="tons_value_seamodes",  # total tons per OD–product–year
    col_fbx="log_avg_fb_rate_usd",
    col_bdi="bdi_avg_usdprice",
    col_gdp_o="origin_gdp_usd_current",
    col_gdp_d="destination_gdp_usd_current",
    col_dist="cepii_dist",
    col_conn="lsbci",
    col_o="origin_isocode",
    col_d="destination_isocode",
    col_year="year",
    col_prod="product_code",
    hs4_bulk_list=("1001","1002","1003","1004","1005","1006","1007","1008","1009"),  # cereals as bulk
    verbose=True  # toggle summary printout
):
    df = df.copy()

    # ---- IDs ----
    if col_o in df.columns and col_d in df.columns:
        df["pair_id"] = df[col_o].astype(str) + "|" + df[col_d].astype(str)
    else:
        raise ValueError("Expected origin/destination ISO code columns not found")

    # ---- Targets ----
    if col_cost_tonkm in df.columns and df[col_cost_tonkm].notna().any():
        df["cost_tonkm_sea"] = df[col_cost_tonkm]
    elif col_exp_usd in df.columns and col_work_tonkm in df.columns:
        df["cost_tonkm_sea"] = df[col_exp_usd] / df[col_tons].replace(0, np.nan)
    else:
        raise ValueError("Need seamode cost or expenditure/work_tonkm")

    df = df[np.isfinite(df["cost_tonkm_sea"]) & (df["cost_tonkm_sea"] > 0)].copy()

    # ---- Logs ----
    for src, dst in [
        (col_gdp_o, "ln_gdp_o"),
        (col_gdp_d, "ln_gdp_d"),
        (col_dist, "ln_dist"),
        (col_bdi, "ln_bdi"),
        ("cost_tonkm_sea", "ln_cost_tonkm"),
    ]:
        df[dst] = _safe_log(df[src].astype(float))

    # ---- Freight indices ----
    if col_fbx not in df.columns or col_bdi not in df.columns:
        raise ValueError("FBX and/or BDI columns missing in dataset")
    df["ln_fbx"] = df[col_fbx].astype(float)

    # ---- Connectivity ----
    df["conn_ij"] = _zscore(df[col_conn].astype(float))
    df["comp_ij"] = df["conn_ij"]

    # ---- Probabilistic bulk share ----
    df["bulk_potential"] = df[col_prod].astype(str).isin(hs4_bulk_list).astype(int)

    # Volume-based scaling (log-ton thresholds)
    a, b = 6.0, 10.0  # in log(tons): exp(6)~403 t, exp(10)~22,026 t

    if col_tons in df.columns:
        vol = df[col_tons].fillna(0).astype(float)
        logv = np.log1p(vol)
        s = (logv - a) / (b - a)
        s = s.clip(lower=0.0, upper=1.0)
        df["bulk_share"] = df["bulk_potential"] * s
    else:
        df["bulk_share"] = df["bulk_potential"].astype(float)

    # ---- Interactions ----
    df["ln_dist_x_bulk"] = df["ln_dist"] * df["bulk_share"]
    df["conn_x_bulk"]    = df["conn_ij"] * df["bulk_share"]
    df["ln_bdi_x_bulk"]  = df["ln_bdi"] * df["bulk_share"]

    # ---- Minimal subset ----
    keep = [
        col_o, col_d,
        "pair_id", col_year, col_prod,
        "cost_tonkm_sea", "ln_cost_tonkm",col_tons,
        "ln_dist", "conn_ij", "comp_ij",
        "ln_gdp_o", "ln_gdp_d",
        "ln_fbx", "ln_bdi", "ln_bdi_x_bulk",
        "bulk_share", "ln_dist_x_bulk", "conn_x_bulk"
    ]
    df_model = df[keep].dropna().copy()

    # ---- Diagnostic summary ----
    if verbose and "bulk_share" in df_model.columns:
        mean_bulk = df_model["bulk_share"].mean()
        corr_vol = (
            df_model["bulk_share"].corr(np.log1p(df[col_tons])) 
            if col_tons in df.columns else np.nan
        )
        corr_bdi = (
            df_model["bulk_share"].corr(df_model["ln_bdi_x_bulk"]) 
            if "ln_bdi_x_bulk" in df_model.columns else np.nan
        )
        print("\n📊 Bulk-share diagnostic summary:")
        print(f"  • Log threshold range: {a:.1f}–{b:.1f} → ≈ {np.exp(a):,.0f}–{np.exp(b):,.0f} tons")
        print(f"  • Mean bulk_share: {mean_bulk:.3f}")
        print(f"  • Corr(bulk_share, log(trade_tons)): {corr_vol:.3f}")
        print(f"  • Corr(bulk_share, ln_bdi_x_bulk): {corr_bdi:.3f}")
        print(f"  • Sample N: {len(df_model):,}")

    return df_model
# ======================================================
# 2) Econometric Models
# ======================================================

def fit_cost_log_ols(df_model: pd.DataFrame, fe_type="none", cluster_on="pair_id"):
    """
    Log-OLS for pooled or pair fixed-effects models.
    fe_type: 'none' -> pooled, 'pair' -> add C(pair_id)
    """
    if fe_type not in ("none", "pair"):
        raise ValueError("fe_type must be 'none' or 'pair'")

    fe_terms = " + C(pair_id)" if fe_type == "pair" else ""

    formula = (
        "ln_cost_tonkm ~ ln_dist + conn_ij + ln_gdp_o + ln_gdp_d "
        "+ ln_fbx + ln_bdi_x_bulk "
        "+ bulk_share + ln_dist_x_bulk + conn_x_bulk" + fe_terms
    )

    y, X = patsy.dmatrices(formula, df_model, return_type="dataframe")
    model = sm.OLS(y, X)
    if cluster_on in df_model.columns:
        res = model.fit(cov_type="cluster", cov_kwds={"groups": df_model[cluster_on]})
    else:
        res = model.fit(cov_type="HC1")

    base_cols = ["ln_dist","conn_ij","ln_gdp_o","ln_gdp_d","ln_fbx","bulk_product","ln_dist_x_bulk","conn_x_bulk"]
    vif = _vif_table(df_model[base_cols].assign(const=1))
    return res, vif


def fit_cost_gamma_glm(df_model: pd.DataFrame, fe_type="none"):
    """
    Gamma GLM (log link) for positive cost data.
    fe_type: 'none' (pooled) or 'pair' (add C(pair_id))
    """
    if fe_type not in ("none", "pair"):
        raise ValueError("fe_type must be 'none' or 'pair'")

    fe_terms = " + C(pair_id)" if fe_type == "pair" else ""

    formula = (
        "cost_tonkm_sea ~ ln_dist + conn_ij + ln_gdp_o + ln_gdp_d"
        "+ ln_fbx + ln_bdi_x_bulk "
        "+ bulk_share + ln_dist_x_bulk + conn_x_bulk" + fe_terms + fe_terms
    )

    y, X = patsy.dmatrices(formula, df_model, return_type="dataframe")
    glm = sm.GLM(y, X, family=sm.families.Gamma(sm.families.links.Log()))
    res = glm.fit(method="lbfgs", maxiter=200, disp=False)
    return res

In [ ]:
dfm_check_feature = build_features(df_bltr_multivar, verbose=True)

## Pooled Model

In [ ]:

# ----------------------------
# 3) Convenience runner
# ----------------------------
def demean_by_group(df, group, cols):
    """Within transformation: subtract group means."""
    df_out = df.copy()
    for col in cols:
        df_out[col] = df[col] - df.groupby(group)[col].transform("mean")
    return df_out

def run_maritime_cost_pipeline(
    df_raw: pd.DataFrame,
    run_ols: bool = True,
    run_glm: bool = True,
    absorb_pair_fe: bool = False,  # <-- toggle FE absorption
    verbose: bool = True
):
    """
    Maritime Transport Cost Model (Pooled or Absorbed FE)
    ------------------------------------------------------
    If absorb_pair_fe=True, applies within-demeaning by pair_id (absorbed FE);
    otherwise, runs a pure pooled model (single global intercept).
    """
    if verbose: print("🔹 Building features...")
    dfm = build_features(df_raw)
    out = {"n_obs": len(dfm), "df_model": dfm}

    # Absorb fixed effects (optional)
    if absorb_pair_fe:
        if verbose: print("⚙️  Applying within-demeaning (absorbing pair FE)...")
        cont_vars = ["ln_cost_tonkm", "ln_dist", "conn_ij", "comp_ij","ln_fbx"
                     "ln_gdp_o", "ln_gdp_d","bulk_share","ln_bdi_x_bulk"
                     "ln_dist_x_bulk", "conn_x_bulk"]
        dfm = demean_by_group(dfm, "pair_id", [v for v in cont_vars if v in dfm.columns])

    # Run OLS
    if run_ols:
        try:
            y = dfm["ln_cost_tonkm"]
            X = dfm[["ln_dist", "conn_ij", "comp_ij", "ln_gdp_o", "ln_gdp_d","ln_fbx",
                     "bulk_share","ln_bdi_x_bulk", "ln_dist_x_bulk", "conn_x_bulk"]].copy()
            X = sm.add_constant(X)
            model = sm.OLS(y, X)
            res = model.fit(cov_type="HC1")
            out.update({
                "ols_success": True,
                "ols_summary": res.summary().as_text(),
                "ols_params": res.params.to_frame("coef")
                                .join(res.bse.rename("se"))
                                .join(res.pvalues.rename("p")),
            })
            if verbose: print("✅ OLS completed successfully.")
        except Exception as e:
            out.update({"ols_success": False, "ols_error": str(e)})
            print("❌ OLS failed:", e)

    # Run Gamma GLM
    if run_glm:
        try:
            y = dfm["cost_tonkm_sea"]
            X = dfm[["ln_dist", "conn_ij", "comp_ij", "ln_gdp_o", "ln_gdp_d","ln_fbx",
                     "bulk_share","ln_bdi_x_bulk", "ln_dist_x_bulk", "conn_x_bulk"]].copy()
            X = sm.add_constant(X)
            glm = sm.GLM(y, X, family=sm.families.Gamma(sm.families.links.Log()))
            resg = glm.fit(method="lbfgs", maxiter=200, disp=False)
            out.update({
                "gamma_success": True,
                "gamma_summary": resg.summary().as_text(),
                "gamma_params": resg.params.to_frame("coef")
                                    .join(resg.bse.rename("se"))
                                    .join(resg.pvalues.rename("p")),
            })
            if verbose: print("✅ Gamma GLM completed successfully.")
        except Exception as e:
            out.update({"gamma_success": False, "gamma_error": str(e)})
            print("❌ Gamma GLM failed:", e)

    if verbose:
        model_type = "Pooled" if not absorb_pair_fe else "Absorbed FE"
        print(f"🏁 {model_type} pipeline finished. N={len(dfm):,}")
    return out

In [ ]:
# Use the function
results = run_maritime_cost_pipeline(df_bltr_multivar, run_ols=True, run_glm=True, absorb_pair_fe=False)
results_fe = run_maritime_cost_pipeline(df_bltr_multivar, run_ols=True, run_glm=True, absorb_pair_fe=True)


In [ ]:
# Extract from OLS model
ols_pooled_df = results["ols_params"].reset_index().rename(columns={
    "index": "variable",
    "coef": "coefficient",
    "se": "std_error",
    "p": "p_value"
})
ols_pooled_df['p_value'] = round(ols_pooled_df['p_value'],3)
ols_pooled_df['model'] = 'OLS'
ols_pooled_df["commodity"] = "Pooled"
ols_pooled_df["r2"] = 0.210   # From your OLS summary above
ols_pooled_df["n_obs"] = results["n_obs"]
structural_vars = [
    "ln_dist", "conn_ij", "comp_ij", "ln_gdp_o", "ln_gdp_d","ln_fbx", "bulk_share","ln_bdi_x_bulk", "ln_dist_x_bulk", "conn_x_bulk"
]
ols_pooled_df = ols_pooled_df[ols_pooled_df["variable"].isin(structural_vars)]

display(ols_pooled_df)

# Extract from GLM model
glm_pooled_df = results["gamma_params"].reset_index().rename(columns={
    "index": "variable",
    "coef": "coefficient",
    "se": "std_error",
    "p": "p_value"
})
glm_pooled_df['p_value'] = round(glm_pooled_df['p_value'],3)
glm_pooled_df['model'] = 'GLM'
glm_pooled_df["commodity"] = "Pooled"
glm_pooled_df["r2"] = 0.03226   # From your OLS summary above
glm_pooled_df["n_obs"] = results["n_obs"]
structural_vars = [
    "ln_dist", "conn_ij", "comp_ij", "ln_gdp_o", "ln_gdp_d","ln_fbx", "bulk_share","ln_bdi_x_bulk", "ln_dist_x_bulk", "conn_x_bulk"
]
glm_pooled_df = glm_pooled_df[glm_pooled_df["variable"].isin(structural_vars)]

display(glm_pooled_df)

## FE Export and Importer

In [ ]:
# ======================================================
# 3) Ferrari-style Country FE Models (Separate)
# ======================================================
def fit_cost_log_ols_country_fe(df_model: pd.DataFrame):
    """
    Log-OLS with exporter/importer fixed effects (Ferrari-style).
    Adds C(origin_isocode) + C(destination_isocode).
    """
    for col in ["origin_isocode", "destination_isocode"]:
        df_model[col] = df_model[col].astype("category")
    for col in df_model.select_dtypes(include="float64").columns:
        df_model[col] = df_model[col].astype("float32")

    formula = (
        "ln_cost_tonkm ~ ln_dist + conn_ij + ln_gdp_o + ln_gdp_d"
        "+ ln_fbx + ln_bdi_x_bulk "
        "+ bulk_share + ln_dist_x_bulk + conn_x_bulk"
        "+ C(origin_isocode, Treatment) + C(destination_isocode, Treatment)"
    )

    y, X = patsy.dmatrices(formula, df_model, return_type="dataframe", sparse=True)
    model = sm.OLS(y, X)
    res = model.fit(cov_type="HC1")
    return res


def fit_cost_gamma_glm_country_fe(df_model: pd.DataFrame):
    """
    Gamma GLM with exporter/importer fixed effects (Ferrari-style).
    """
    for col in ["origin_isocode", "destination_isocode"]:
        df_model[col] = df_model[col].astype("category")
    for col in df_model.select_dtypes(include="float64").columns:
        df_model[col] = df_model[col].astype("float32")

    formula = (
        "cost_tonkm_sea ~ ln_dist + conn_ij + ln_gdp_o + ln_gdp_d"
        "+ ln_fbx + ln_bdi_x_bulk "
        "+ bulk_share + ln_dist_x_bulk + conn_x_bulk"
        "+ C(origin_isocode, Treatment) + C(destination_isocode, Treatment)"
    )

    y, X = patsy.dmatrices(formula, df_model, return_type="dataframe", sparse=True)
    glm = sm.GLM(y, X, family=sm.families.Gamma(sm.families.links.Log()))
    res = glm.fit(method="lbfgs", maxiter=200, disp=False)
    return res
def evaluate_model_performance_batch(res, dfm, y_col="ln_cost_tonkm", extreme_thresh=0.4):
    """
    Compute RMSE, MAPE, Bias, and extreme residual count for a single FE model.
    Assumes y_col is in logs (ln cost per ton).
    """
    dfp = dfm.copy()

    # True and predicted in log
    dfp["y_true"] = dfp[y_col]
    dfp["y_pred"] = res.predict()  # aligned with patsy design

    # Back to levels
    dfp["true_level"] = np.exp(dfp["y_true"])
    dfp["pred_level"] = np.exp(dfp["y_pred"])

    # Errors (in levels)
    dfp["abs_error"] = np.abs(dfp["pred_level"] - dfp["true_level"])
    dfp["perc_error"] = dfp["abs_error"] / np.clip(dfp["true_level"], 1e-9, None)

    # Log residuals
    dfp["resid_log"] = dfp["y_true"] - dfp["y_pred"]
    dfp["abs_resid_log"] = np.abs(dfp["resid_log"])

    # Metrics
    rmse = np.sqrt(np.mean((dfp["pred_level"] - dfp["true_level"])**2))
    mape = float(dfp["perc_error"].mean() * 100)
    bias = float((dfp["pred_level"] - dfp["true_level"]).mean())
    extreme_mask = dfp["perc_error"] > extreme_thresh
    extreme_count = int(extreme_mask.sum())
    extreme_share = 100 * extreme_count / len(dfp)

    metrics = {
        "RMSE": rmse,
        "MAPE(%)": mape,
        "Bias": bias,
        "ExtremeResiduals": extreme_count,
        "ExtremeShare(%)": extreme_share,
    }

    return metrics, dfp, dfp[extreme_mask].copy()

In [ ]:
import gc, statsmodels.api as sm, pandas as pd, numpy as np, traceback, os
from tqdm import tqdm

def run_maritime_cost_fe_country_optimized(
    df_raw: pd.DataFrame,
    commodity_code: str,
    output_dir: str = "results_fe",
    extreme_thresh: float = 2.0,
    run_ols: bool = True,
    verbose: bool = True,
):
    """
    Ferrari-style fixed exporter/importer FE model (optimized for limited RAM)
    --------------------------------------------------------------------------
    Runs one commodity at a time with exporter/importer fixed effects.
    """
    os.makedirs(output_dir, exist_ok=True)

    # ---- 1. Prepare subset ----
    print(f"🔹 {commodity_code}")
    dfm = build_features(df_raw)
    dupes = dfm.index[dfm.index.duplicated()]
    print("Duplicate index entries:", len(dupes))
    dfm = dfm[dfm["product_code"].astype(str) == str(commodity_code)].copy()
    if len(dfm) < 100:
        if verbose:
            print(f"⚠️ {commodity_code}: not enough observations ({len(dfm)})")
        return None

    # Reduce memory footprint
    for col in dfm.select_dtypes(include="float64"):
        dfm[col] = dfm[col].astype("float32")
    for catcol in ["origin_isocode", "destination_isocode"]:
        dfm[catcol] = dfm[catcol].astype("category")

    if verbose:
        print(f"🔹 {commodity_code}: N={len(dfm):,}, "
              f"Exporters={dfm['origin_isocode'].nunique()}, "
              f"Importers={dfm['destination_isocode'].nunique()}")

    out = {"commodity": commodity_code, "n_obs": len(dfm)}

    # ---- 2. OLS with country FE ----
    if run_ols:
        try:
            # build formula (no sparse argument)
            formula = (
                "ln_cost_tonkm ~ ln_dist + conn_ij + ln_gdp_o + ln_gdp_d"
                "+ ln_fbx + ln_bdi_x_bulk "
                "+ bulk_share + ln_dist_x_bulk + conn_x_bulk"
                "+ C(origin_isocode, Treatment) + C(destination_isocode, Treatment)"
            )
            y, X = patsy.dmatrices(formula, dfm, return_type="dataframe")  # <--- removed sparse=True
            model = sm.OLS(y, X)
            res = model.fit(cov_type="HC1")

            # 2a. Save summaries
            txt_path = os.path.join(output_dir, f"{commodity_code}_OLS_summary.txt")
            csv_path = os.path.join(output_dir, f"{commodity_code}_OLS_params.csv")
            with open(txt_path, "w") as f:
                f.write(res.summary().as_text())

            res.params.to_frame("coef") \
                .join(res.bse.rename("se")) \
                .join(res.pvalues.rename("p")) \
                .to_csv(csv_path)

            # 2b. Diagnostics + predictions (full + extremes)
            metrics, df_pred, df_extreme = evaluate_model_performance_batch(
                res, dfm, y_col="ln_cost_tonkm", extreme_thresh=extreme_thresh
            )

            # merge metrics into out
            out.update({"r2": float(res.rsquared)})
            out.update(metrics)

            # 2c. Save diagnostics tables
            diag_path = os.path.join(output_dir, f"{commodity_code}_Diagnostics.csv")
            pd.DataFrame([{
                "commodity": commodity_code,
                "R2": res.rsquared,
                **metrics
            }]).to_csv(diag_path, index=False)

            # 2d. Save predictions
            # keep only relevant identifiers + error columns
            pred_keep = [
                "origin_isocode", "destination_isocode", "product_code", "year",
                "tons_value_seamodes","","ln_cost_tonkm", "y_pred",
                "true_level", "pred_level",
                "abs_error", "perc_error", "resid_log", "abs_resid_log","ln_dist"
            ]
            pred_keep = [c for c in pred_keep if c in df_pred.columns]
            preds_path = os.path.join(output_dir, f"{commodity_code}_Predictions.csv")
            df_pred[pred_keep].to_csv(preds_path, index=False)

            # 2e. Save only extreme residuals
            if len(df_extreme) > 0:
                ext_path = os.path.join(output_dir, f"{commodity_code}_ExtremeResiduals.csv")
                df_extreme[pred_keep].to_csv(ext_path, index=False)

            if verbose:
                print(
                    f"   ✅ R²={res.rsquared:.3f}, "
                    f"RMSE={metrics['RMSE']:.4f}, "
                    f"MAPE={metrics['MAPE(%)']:.2f}%, "
                    f"Extremes={metrics['ExtremeResiduals']} "
                    f"({metrics['ExtremeShare(%)']:.2f}%)"
                )
                
        except Exception as e:
            print(f"❌ {commodity_code}: OLS failed - {e}")
            traceback.print_exc(limit=1)
            out.update({"error": str(e)})

    # ---- 3. Memory cleanup ----
    if "res" in locals():
        del res
    del dfm, X, y
    gc.collect()
    return out


def batch_run_fe_models(
    df_raw: pd.DataFrame,
    output_dir: str = "results_fe",
    product_list: list | None = None,
    resume: bool = True,
    verbose: bool = True,
):
    """
    Batch runner for Ferrari-style FE models (by commodity).
    Includes progress bar, resume, and incremental saving.
    """
    os.makedirs(output_dir, exist_ok=True)

    # ---- Collect product list ----
    if product_list is None:
        product_list = sorted(df_raw["product_code"].unique().astype(str))

    # ---- Detect completed commodities ----
    completed = set()
    if resume:
        completed = {
            f.split("_")[0]
            for f in os.listdir(output_dir)
            if f.endswith("_OLS_summary.txt")
        }
        if verbose:
            print(f"🔍 Resuming: {len(completed)} commodities already processed.")

    summaries = []
    for prod in tqdm(product_list, desc="Running FE models by commodity"):
        if prod in completed:
            if verbose:
                print(f"⏩ Skipping {prod} (already done).")
            continue

        res = run_maritime_cost_fe_country_optimized(
            df_raw, prod, output_dir=output_dir, run_ols=True, verbose=verbose
        )
        if res:
            summaries.append(res)
            # Save incremental progress
            pd.DataFrame(summaries).to_csv(
                os.path.join(output_dir, "FE_summary_progress.csv"),
                index=False
            )
        gc.collect()

    # ---- Final summary save ----
    if summaries:
        summary_df = pd.DataFrame(summaries)
        summary_df.to_csv(os.path.join(output_dir, "FE_summary_overview.csv"), index=False)
        print(f"✅ Completed all. Summary saved to {output_dir}/FE_summary_overview.csv")
        return summary_df
    else:
        print("⚠️ No new commodities were run.")
        return None
results_fe_commodity = batch_run_fe_models(
    df_bltr_multivar,
    output_dir="results_fe",
    resume=True
)

In [ ]:
import os, re, pandas as pd

def build_fe_comparison_table_detailed(output_dir="results_fe"):
    """
    Builds a cross-commodity comparison table of Ferrari-style FE models.
    Filters out exporter/importer dummy variables and merges with R² overview.
    """

    # ---- Load R² overview ----
    overview_path = os.path.join(output_dir, "FE_summary_overview.csv")
    if not os.path.exists(overview_path):
        raise FileNotFoundError(f"Missing FE_summary_overview.csv in {output_dir}")
    overview = pd.read_csv(overview_path)

    # ---- Ensure commodity is string for merge ----
    overview["commodity"] = overview["commodity"].astype(str)

    # ---- Collect OLS param files ----
    records = []
    param_files = [f for f in os.listdir(output_dir) if f.endswith("_OLS_params.csv")]

    structural_vars = [
        "ln_dist", "conn_ij", "ln_gdp_o", "ln_gdp_d", 'ln_fbx',
        "ln_bdi_x_bulk", "bulk_share", "ln_dist_x_bulk", "conn_x_bulk"
    ]

    for file in param_files:
        commodity = re.findall(r"([0-9]+)_OLS_params", file)[0]
        params = pd.read_csv(os.path.join(output_dir, file))

        # Ensure variable names are present
        if "Unnamed: 0" in params.columns:
            params = params.rename(columns={"Unnamed: 0": "variable"})
        elif "index" not in params.columns:
            params = params.reset_index().rename(columns={"index": "variable"})

        # Convert variable names to string
        params["variable"] = params["variable"].astype(str)

        # Keep only relevant structural variables
        params = params[params["variable"].isin(structural_vars)].copy()

        # Add commodity column
        params["commodity"] = commodity

        # Keep core columns (skip if missing)
        for col in ["coef", "p", "se"]:
            if col not in params.columns:
                params[col] = None

        records.append(params[["commodity", "variable", "coef", "se", "p"]])

    if not records:
        print("⚠️ No parameter files found in directory.")
        return None

    # ---- Combine all coefficients ----
    df_all = pd.concat(records, ignore_index=True)

    # ---- Merge with R² overview ----
    comp = df_all.merge(overview.astype({"commodity": str}), on="commodity", how="left")

    # ---- Clean up columns ----
    comp = comp.rename(columns={
        "coef": "coefficient",
        "p": "p_value",
        "se": "std_error"
    })[["commodity", "variable", "coefficient", "std_error", "p_value", "r2", "n_obs"]]

    # ---- Sort nicely ----
    comp = comp.sort_values(["variable", "commodity"]).reset_index(drop=True)

    # ---- Save output ----
    output_path = os.path.join(output_dir, "FE_comparison_elasticities.csv")
    comp.to_csv(output_path, index=False)
    print(f"✅ Detailed comparison table saved to {output_path}")

    return comp
comparison_df = build_fe_comparison_table_detailed("results_fe")


In [ ]:
summary_stats = (
    comparison_df.groupby("variable")["coefficient"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
).reset_index()
display(summary_stats)
summary_stats_allvar = (
    comparison_df.groupby(["variable"])
    .agg({'coefficient' : 'mean', 'p_value' : 'mean'})
    .round(3)
).reset_index()
display(summary_stats_allvar)

### Optional comparison requiring an additional object

The original cell depends on `pooled_df`, which is not constructed in this notebook. Retained below for reference, outside the execution path.

```python
# Assuming comparison_df already loaded from build_fe_comparison_table_detailed()
combined = pd.concat([comparison_df, pooled_df], ignore_index=True)

# Sort neatly by variable and commodity
combined = combined.sort_values(["variable", "commodity"]).reset_index(drop=True)

pivot_compare = (
    combined
    .pivot_table(index="variable", columns="commodity",
                 values="coefficient")
    .round(3)
)

display(pivot_compare)
```


### Optional comparison requiring an additional object

The original cell depends on `pooled_df`, which is not constructed in this notebook. Retained below for reference, outside the execution path.

```python
# Average of country-FE elasticities (excluding pooled)
avg_country = (
    comparison_df.groupby("variable")["coefficient"]
    .mean()
    .rename("country_FE_mean")
)

pooled = pooled_df.set_index("variable")["coefficient"].rename("pooled")

compare_elasticities = pd.concat([pooled, avg_country], axis=1).round(3)
compare_elasticities["diff"] = compare_elasticities["country_FE_mean"] - compare_elasticities["pooled"]

display(compare_elasticities)
```


## Econometric Model - Evaluation

In [ ]:
import numpy as np
import pandas as pd

def evaluate_pooled_models(results, ols=True, glm=True, extreme_thresh=2.0):
    """
    Compute R², RMSE, MAPE, Bias, and Extreme Residual share for pooled OLS and/or GLM models.
    Parameters
    ----------
    results : dict
        Output from run_maritime_cost_pipeline()
    ols : bool
        Whether to compute diagnostics for OLS
    glm : bool
        Whether to compute diagnostics for Gamma GLM
    extreme_thresh : float
        Threshold multiplier of RMSE for defining extreme residuals
    """
    out = {}

    # ======================================================
    # 1️⃣ OLS diagnostics
    # ======================================================
    if ols and results.get("ols_success", False):
        df = results["df_model"].copy()
        y_true = df["ln_cost_tonkm"].values
        model_ols = sm.OLS(y_true, sm.add_constant(df[[
            "ln_dist", "conn_ij", "comp_ij", "ln_gdp_o", "ln_gdp_d","ln_fbx",
                     "bulk_share","ln_bdi_x_bulk", "ln_dist_x_bulk", "conn_x_bulk"
        ]])).fit(cov_type="HC1")

        y_pred = model_ols.fittedvalues.values
        residuals = y_true - y_pred

        r2 = model_ols.rsquared
        rmse = np.sqrt(np.mean(residuals**2))
        mape = np.mean(np.abs(residuals / y_true)) * 100
        bias = np.mean(residuals)
        abs_pct_error = np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))
        extreme_mask = abs_pct_error > 0.3  # >30%
        extreme_count = np.sum(extreme_mask)
        extreme_share = extreme_count / len(df) * 100

        out["OLS"] = {
            "R2": r2,
            "RMSE": rmse,
            "MAPE(%)": mape,
            "Bias": bias,
            "ExtremeResiduals": int(extreme_count),
            "ExtremeShare(%)": extreme_share,
            "N_obs": len(df)
        }

    # ======================================================
    # 2️⃣ GLM diagnostics
    # ======================================================
    if glm and results.get("gamma_success", False):
        df = results["df_model"].copy()
        y_true = df["cost_tonkm_sea"].values
        X = sm.add_constant(df[[
            "ln_dist", "conn_ij", "comp_ij", "ln_gdp_o", "ln_gdp_d","ln_fbx",
                     "bulk_share","ln_bdi_x_bulk", "ln_dist_x_bulk", "conn_x_bulk"
        ]])
        model_glm = sm.GLM(y_true, X, family=sm.families.Gamma(sm.families.links.Log())).fit()

        y_pred = model_glm.fittedvalues.values
        residuals = y_true - y_pred

        # Pseudo-R² for GLM (Cragg–Uhler / McFadden approximation)
        r2 = 1 - (model_glm.deviance / model_glm.null_deviance)
        rmse = np.sqrt(np.mean(residuals**2))
        mape = np.mean(np.abs(residuals / y_true)) * 100
        bias = np.mean(residuals)
        abs_pct_error = np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))
        extreme_mask = abs_pct_error > 0.3  # >30%
        extreme_count = np.sum(extreme_mask)
        extreme_share = extreme_count / len(df) * 100

        out["GLM"] = {
            "R2": r2,
            "RMSE": rmse,
            "MAPE(%)": mape,
            "Bias": bias,
            "ExtremeResiduals": int(extreme_count),
            "ExtremeShare(%)": extreme_share,
            "N_obs": len(df)
        }

    return pd.DataFrame(out).T.round(4)
pooled_metrics = evaluate_pooled_models(results, ols=True, glm=True)
display(pooled_metrics)

In [ ]:
import pandas as pd
import glob, os

def merge_fe_diagnostics(output_dir="results_fe"):
    """
    Merge all *_Diagnostics.csv from the FE model output folder.
    Recalculates global ExtremeShare across all observations.
    Returns combined DataFrame sorted by R2 descending.
    """
    files = glob.glob(os.path.join(output_dir, "*_Diagnostics.csv"))
    if not files:
        print("⚠️ No diagnostic files found.")
        return None

    # Read and concatenate
    df_list = []
    for f in files:
        tmp = pd.read_csv(f)
        tmp["commodity"] = os.path.basename(f).split("_")[0]
        df_list.append(tmp)
    all_diag = pd.concat(df_list, ignore_index=True)

    # Standardize numeric columns
    for c in ["R2", "RMSE", "MAPE(%)", "Bias", "ExtremeResiduals", "ExtremeShare(%)"]:
        if c in all_diag.columns:
            all_diag[c] = pd.to_numeric(all_diag[c], errors="coerce")

    # Compute overall totals
    total_obs = 0
    total_extremes = 0
    for f in files:
        preds_file = f.replace("_Diagnostics.csv", "_Predictions.csv")
        if os.path.exists(preds_file):
            df_preds = pd.read_csv(preds_file, usecols=["perc_error"])
            total_obs += len(df_preds)
            total_extremes += (df_preds["perc_error"] > 0.4).sum()

    global_extreme_share = 100 * total_extremes / total_obs if total_obs > 0 else None

    print("✅ Merged diagnostics summary:")
    print(f"   Commodities: {len(all_diag)}")
    print(f"   Total observations: {total_obs:,}")
    print(f"   Total extreme residuals (|perc_error|>0.4): {total_extremes:,}")
    print(f"   Global extreme share: {global_extreme_share:.2f}%")

    # Rank and save
    summary_path = os.path.join(output_dir, "FE_Diagnostics_Merged.csv")
    all_diag.sort_values("R2", ascending=False, inplace=True)
    all_diag.to_csv(summary_path, index=False)

    print(f"📄 Saved merged diagnostics to {summary_path}")
    return all_diag, global_extreme_share
all_diag, global_extreme_share = merge_fe_diagnostics("results_fe")

In [ ]:
def merge_fe_prediction(output_dir="results_fe"):
    """
    Merge all *_prediction.csv from the FE model output folder.
    for deep dive of detail level
    """
    files = glob.glob(os.path.join(output_dir, "*_Predictions.csv"))
    df_list = []
    for f in files:
        tmp = pd.read_csv(f)
        tmp["commodity"] = os.path.basename(f).split("_")[0]
        df_list.append(tmp)
    all_pred = pd.concat(df_list, ignore_index=True)    
    prediction_path = os.path.join(output_dir, "FE_predictions_Merged.csv")
    all_pred.to_csv(prediction_path, index=False) 
    print(f"📄 Saved merged predictions dataframe to {prediction_path}")
    return all_pred
df_fe_model_prediction = merge_fe_prediction("results_fe")
df_fe_model_prediction['year_count'] = (
    df_fe_model_prediction.groupby(['origin_isocode', 'destination_isocode','product_code'])['year']
      .transform('nunique')
)
df_fe_model_prediction['is_extreme'] = (df_fe_model_prediction["perc_error"] > 0.5).astype(int)

In [ ]:
import pandas as pd
import glob, os

def summarize_distance_elasticity(output_dir="results_fe"):
    """
    Collects ln_dist coefficient from each commodity OLS result file.
    Returns a summary DataFrame with mean, std, and comparison stats.
    """
    files = glob.glob(os.path.join(output_dir, "*_OLS_params.csv"))
    records = []

    for f in files:
        df = pd.read_csv(f, index_col=0)
        if "ln_fbx" in df.index:
            commodity = os.path.basename(f).split("_")[0]
            coef = df.loc["ln_fbx", "coef"]
            se = df.loc["ln_fbx", "se"]
            pval = df.loc["ln_fbx", "p"]
            records.append({"commodity": commodity, "beta_fbx": coef, "se": se, "pval": pval})

    dist_df = pd.DataFrame(records)
    dist_df["significant"] = dist_df["pval"] < 0.05
    mean_elasticity = dist_df["beta_fbx"].mean()
    std_elasticity = dist_df["beta_fbx"].std()

    print(f"📘 Mean fbx elasticity: {mean_elasticity:.3f}")
    print(f"σ (std dev): {std_elasticity:.3f}")
    print(f"Significant in {dist_df['significant'].sum()}/{len(dist_df)} commodities")

    return dist_df.sort_values("beta_fbx"), mean_elasticity, std_elasticity
fbx_df, mean_elast, std_elast = summarize_distance_elasticity("results_fe")

## Machine Learning

### Ridge Regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer

In [ ]:
# Helper: Evaluation Metrics
# ==========================================================
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    return r2, rmse, mae, mape

# ==========================================================
# 1. Load DataFrame (replace this line in your notebook)
# ==========================================================
# from your_script import build_features
df_ml_features = build_features(df_bltr_multivar)
print("⚠️ Please replace this with your actual DataFrame loading + build_features()")
# Example:
# from df_bltr_multivar import df_bltr_multivar as df
# df_features = build_features(df)

# ==========================================================
# 2. Prepare Data
# ==========================================================
target_var = "ln_cost_tonkm"

feature_cols = [
    "ln_gdp_o", "ln_gdp_d", "ln_dist", "conn_ij",
    "ln_bdi", "ln_fbx", "ln_dist_x_bulk",
    "conn_x_bulk", "ln_bdi_x_bulk", "bulk_share"
]

categorical_cols = ["origin_isocode", "destination_isocode",'product_code']
##extra_cols = ["product_code"]

df_ml_model = df_ml_features.dropna(subset=[target_var] + feature_cols + categorical_cols).copy()
X_num = df_ml_model[feature_cols]
X_cat = df_ml_model[categorical_cols]
##x_extra = df_ml_model[extra_cols]
y = df_ml_model[target_var]

# ==========================================================
# 3. Preprocessing Pipeline
# ==========================================================
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), feature_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
])

# ==========================================================
# 4. Train-Test Split
# ==========================================================
X_all = pd.concat([X_num, X_cat], axis=1)


def groupwise_stratified_split(X, y, group_col, test_size=0.7, random_state=42):
    train_idx, test_idx = [], []

    for group in X[group_col].unique():
        group_idx = X[X[group_col] == group].index
        g_train, g_test = train_test_split(
            group_idx, test_size=test_size, random_state=random_state
        )
        train_idx.extend(g_train)
        test_idx.extend(g_test)

    return (
        X.loc[train_idx], X.loc[test_idx],
        y.loc[train_idx], y.loc[test_idx]
    )
ridge_X_train, ridge_X_test, ridge_y_train, ridge_y_test = groupwise_stratified_split(X_all, y, group_col="product_code")
# train_years = [2016, 2017, 2018, 2019]
# test_years = [2020, 2021]

# ridge_df_train = df_ann_model[df_ann_model["year"].isin(train_years)].copy()
# ridge_df_test  = df_ann_model[df_ann_model["year"].isin(test_years)].copy()

# ridge_X_train = ridge_df_train[feature_cols + categorical_cols]
# ridge_y_train = ridge_df_train["ln_cost_tonkm"]
# ridge_X_test  = ridge_df_test[feature_cols + categorical_cols]
# ridge_y_test  = ridge_df_test["ln_cost_tonkm"]
# ==========================================================
# 5. Ridge Regression Pipeline
# ==========================================================

ridge_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("ridge", RidgeCV(alphas=np.logspace(-4, 4, 25), cv=5))
])

ridge_pipeline.fit(ridge_X_train, ridge_y_train)
ridge_y_pred = ridge_pipeline.predict(ridge_X_test)

# ==========================================================
# 6. Evaluate Results
# ==========================================================
ridge_r2, ridge_rmse, ridge_mae, ridge_mape = compute_metrics(ridge_y_test, ridge_y_pred)

print("\n📊 Ridge Regression (with One-Hot FE) Performance")
print(f"R²     : {ridge_r2:.4f}")
print(f"RMSE   : {ridge_rmse:.4f}")
print(f"MAE    : {ridge_mae:.4f}")
print(f"MAPE   : {ridge_mape:.2f}%")

#Save Pipeline
import joblib
joblib.dump(ridge_pipeline, "ridge_model_pipeline.pkl")

ridge_pred_df = ridge_X_test.copy()
ridge_pred_df["y_true"] = ridge_y_test
ridge_pred_df["y_pred"] = ridge_y_pred
ridge_pred_df.to_csv("ridge_fe_predictions_ln_cost_tonkm.csv", index=False)
print("✅ Saved prediction results to ridge_ohe_predictions_ln_cost_tonkm.csv")

# Extract coefficients
ridge_coefs = ridge_pipeline.named_steps["ridge"].coef_

# Get feature names after preprocessing
ridge_feature_names = ridge_pipeline.named_steps["preprocess"].get_feature_names_out()



In [ ]:
import re
# Initialize output
records = []

for name, coef in zip(ridge_feature_names, ridge_coefs):
    if name.startswith("cat__"):
        # e.g. "cat__origin_isocode_MLI"
        match = re.match(r"cat__([a-z_]+)_(.+)", name)
        if match:
            category = match.group(1)
            variable = match.group(2)
        else:
            category = "unknown"
            variable = name
    elif name.startswith("num__"):
        # e.g. "num__ln_bdi"
        category = "numerical"
        variable = name.replace("num__", "")
    else:
        category = "numerical"
        variable = name

    records.append({
        "category": category,
        "variable": variable,
        "coefficient": round(coef, 5)
    })

# Create DataFrame
ridge_coef_df = pd.DataFrame(records).sort_values("coefficient", ascending=False)

ridge_params = ridge_pipeline.named_steps["ridge"].get_params()
# RIDGE
for k, v in ridge_params.items():
    print(f"{k}: {v}")
ridge_pipeline.named_steps["preprocess"].transformers

### ANN

**Publication review note.** Although year-based train/test variables are created, GridSearchCV is fitted on ann_X/ann_y (all data), then evaluated on that same data. These reported scores are in-sample.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import shap

# ==========================================================
# 1. Helper Functions
# ==========================================================
def ann_compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    return r2, rmse, mae, mape

# ==========================================================
# 2. Load Data (adjust this to your notebook)
# ==========================================================
df_ann_features = build_features(df_bltr_multivar)

target_ann = "ln_cost_tonkm"
feature_ann_cols = [
    "ln_gdp_o", "ln_gdp_d", "ln_dist", "conn_ij",
    "ln_bdi", "ln_fbx", "ln_dist_x_bulk",
    "conn_x_bulk", "ln_bdi_x_bulk", "bulk_share"
]
categorical_ann_cols = ["origin_isocode", "destination_isocode", "product_code"]

df_ann_model = df_ann_features.dropna(subset=[target_ann] + feature_ann_cols + categorical_ann_cols).copy()
df_ann_model = df_ann_model.sort_values("year")  # Ensure time order

train_years = [2016, 2017, 2018, 2019]
test_years = [2020, 2021]

ann_df_train = df_ann_model[df_ann_model["year"].isin(train_years)].copy()
ann_df_test  = df_ann_model[df_ann_model["year"].isin(test_years)].copy()

ann_X_train = ann_df_train[feature_ann_cols + categorical_ann_cols]
ann_y_train = ann_df_train["ln_cost_tonkm"]
ann_X_test  = ann_df_test[feature_ann_cols + categorical_ann_cols]
ann_y_test  = ann_df_test["ln_cost_tonkm"]

ann_X_num = df_ann_model[feature_ann_cols]
ann_X_cat = df_ann_model[categorical_ann_cols]
ann_y = df_ann_model[target_ann]
ann_X = pd.concat([ann_X_num, ann_X_cat], axis=1)

# ==========================================================
# 3. Preprocessing
# ==========================================================
ann_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), feature_ann_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_ann_cols)
])

# ==========================================================
# 4. TimeSeries Split
# ==========================================================
tscv = TimeSeriesSplit(n_splits=3)

ann_model = Pipeline([
    ("preprocess", ann_preprocessor),
    ("ann", MLPRegressor(
        activation="relu",
        solver="adam",
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    ))
])

ann_grid = {
    "ann__hidden_layer_sizes": [(128, 64), (128, 64, 32)],
    "ann__alpha": [0.0001, 0.001],
    "ann__learning_rate_init": [0.001],
    "ann__max_iter": [500]
}

grid_ann = GridSearchCV(ann_model, ann_grid, cv=tscv, scoring="r2", verbose=1, n_jobs=-1)
grid_ann.fit(ann_X, ann_y)

print(f"✅ Best ANN Params: {grid_ann.best_params_}")
ann_y_pred = grid_ann.predict(ann_X)
ann_r2, ann_rmse, ann_mae, ann_mape = ann_compute_metrics(ann_y, ann_y_pred)
# Save pipeline
import joblib
joblib.dump(grid_ann.best_estimator_, "ann_model_pipeline.pkl")

print("\n📊 ANN Performance (All Data After Tuning)")
print(f"R²     : {ann_r2:.4f}")
print(f"RMSE   : {ann_rmse:.4f}")
print(f"MAE    : {ann_mae:.4f}")
print(f"MAPE   : {ann_mape:.2f}%")

# Save predictions
ann_pred_df = ann_X.copy()
ann_pred_df["year"] = df_ann_model["year"].values
ann_pred_df["y_true"] = ann_y
ann_pred_df["y_pred"] = ann_y_pred
ann_pred_df.to_csv("ann_timesplit_predictions_ln_cost_tonkm.csv", index=False)

# ==========================================================
# 5. Plot Loss Curve
# ==========================================================
plt.plot(grid_ann.best_estimator_.named_steps["ann"].loss_curve_)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve – Best ANN")
plt.grid(True)
plt.tight_layout()
plt.savefig("ann_timesplit_training_loss.png", dpi=300)

# ==========================================================
# 6. SHAP Interpretability
# ==========================================================
print("🧠 Running SHAP analysis on ANN...")

ann_X_trans = grid_ann.best_estimator_.named_steps["preprocess"].transform(ann_X)
ann_best_model = grid_ann.best_estimator_.named_steps["ann"] #####Get the ANN model only (after preprocessing)
ann_explainer = shap.KernelExplainer(ann_best_model.predict, ann_X_trans[:50])
ann_shap_values = ann_explainer.shap_values(ann_X_trans[:50])
ann_feature_names = grid_ann.best_estimator_.named_steps["preprocess"].get_feature_names_out()
shap.summary_plot(ann_shap_values, ann_X_trans[:50], feature_names=ann_feature_names, show=False)
plt.tight_layout()
plt.savefig("shap_ann_timesplit_summary.png", dpi=300)
print("📈 SHAP summary saved.")

In [ ]:
# SHAP Plot for Numerical Features only
# 2. Identify indices of numerical features
num_indices = [i for i, name in enumerate(ann_feature_names) if name.startswith("num__")]

# 3. Filter SHAP values and feature data for numerical features only
ann_X_num_trans = ann_X_trans[:50, num_indices]
ann_shap_num_values = ann_shap_values[:, num_indices]  # already 50 rows
ann_num_feature_names = [ann_feature_names[i] for i in num_indices]

# 4. Plot SHAP summary plot for numerical features
shap.summary_plot(ann_shap_num_values, ann_X_num_trans, feature_names=ann_num_feature_names, show=False)

plt.tight_layout()

In [ ]:
import re

# Compute mean absolute SHAP value across 50 samples
mean_abs_shap = ann_shap_values.mean(axis=0)

# Prepare records
records = []

for name, shap_val in zip(ann_feature_names, mean_abs_shap):
    if name.startswith("cat__"):
        match = re.match(r"cat__([a-z_]+)_(.+)", name)
        if match:
            category = match.group(1)
            variable = match.group(2)
        else:
            category = "unknown"
            variable = name
    elif name.startswith("num__"):
        category = "numerical"
        variable = name.replace("num__", "")
    else:
        category = "numerical"
        variable = name

    records.append({
        "category": category,
        "variable": variable,
        "mean_abs_SHAP": round(shap_val, 5)
    })

# Create DataFrame
ann_shap_df = pd.DataFrame(records).sort_values("mean_abs_SHAP", ascending=False)

# Optional: Save to CSV
ann_shap_df.to_csv("ann_shap_feature_ranking.csv", index=False)

## ML model evaluation

In [ ]:
def evaluate_ml_model(y_true, y_pred, label="Model"):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100

    # Extreme error analysis
    abs_pct_error = np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))
    extreme_mask = abs_pct_error > 0.3  # >30%
    extreme_count = extreme_mask.sum()
    total = len(y_true)
    extreme_share = extreme_count / total * 100

    return {
        "Model": label,
        "R2": round(r2, 4),
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "MAPE (%)": round(mape, 2),
        "Extreme Errors (>30%)": int(extreme_count),
        "Extreme Share (%)": round(extreme_share, 2)
    }

In [ ]:
y_ridge = ridge_pred_df['y_true']
y_pred_ridge = ridge_pred_df['y_pred']
y_ann = ann_pred_df['y_true']
y_pred_ann = ann_pred_df['y_pred']

# ANN
metrics_ann = evaluate_ml_model(y_ann, y_pred_ann, label="ANN")

# Ridge
metrics_ridge = evaluate_ml_model(y_ridge, y_pred_ridge, label="Ridge Regression")
df_evaluation_ann = pd.DataFrame([metrics_ann])
df_evaluation_ridge = pd.DataFrame([metrics_ridge])

# Forecast Modellig: Trade Flow

In [ ]:
base_predictors = [
    # Gravity (size + distance)
    "Log_Origin_GDP",
    "Log_Destination_GDP",
    "Log_Distance",           # from cepii_dist
    "cepii_contig",
    "cepii_comlang_ethno",
    "cepii_wto_o",
    "cepii_wto_d",
    "cepii_fta_wto",

    # Maritime cost & connectivity
    "Log_cost_per_ton_seamodes",   # or Log_Transport_Cost if you prefer
    "lsbci",

    # Food supply/demand
    "Log_origin_production_qty",
    "Log_destination_production_qty",
    "Log_Origin_Daily_Calorific_Food",
    "Log_Destination_Daily_Calorific_Food",
]

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import patsy
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------
# Evaluation helper
# ---------------------------
def evaluate_predictions(y_true, y_pred):
    """RMSE, MAE, MAPE, Bias, Extreme error share."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    abs_err = np.abs(y_true - y_pred)
    bias = np.mean(y_pred - y_true)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    denom = np.where(y_true == 0, np.nan, y_true)
    ape = np.abs((y_true - y_pred) / denom)
    mape = np.nanmean(ape) * 100
    extreme_share = np.nanmean(ape > 0.30) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Bias": bias,
        "Extreme >30% (%)": extreme_share
    }

# ---------------------------
# PPML pseudo R²
# ---------------------------
def pseudo_r2_mcfadden(glm_result, y):
    y_vec = np.asarray(y, dtype=float)
    X_null = np.ones((len(y_vec), 1))
    null_fit = sm.GLM(y_vec, X_null, family=sm.families.Poisson()).fit()
    return 1.0 - glm_result.llf / null_fit.llf

# ---------------------------
# Core maritime prep
# ---------------------------
def prepare_maritime_core(df, predictors, eps=1e-6):
    """
    Prepare data for maritime trade regressions.
    - Ensures logs exist.
    - Drops rows with NaN/inf in necessary columns.
    """
    dfc = df.copy()

    # Dependent variables must exist
    if "Log_Sea_Trade_Volume_Tons" not in dfc.columns:
        dfc["Log_Sea_Trade_Volume_Tons"] = np.log(
            np.where(dfc["tons_value_seamodes"] > 0, dfc["tons_value_seamodes"], eps)
        )

    # Make sure predictors exist & numeric
    predictors = [p for p in predictors if p in dfc.columns]
    cols_needed = predictors + ["Log_Sea_Trade_Volume_Tons", "tons_value_seamodes"]

    dfc = dfc.replace([np.inf, -np.inf], np.nan)
    dfc = dfc.dropna(subset=cols_needed)

    dfc[predictors] = dfc[predictors].apply(pd.to_numeric, errors="coerce")
    dfc["Log_Sea_Trade_Volume_Tons"] = pd.to_numeric(dfc["Log_Sea_Trade_Volume_Tons"], errors="coerce")
    dfc["tons_value_seamodes"] = pd.to_numeric(dfc["tons_value_seamodes"], errors="coerce")

    dfc = dfc.dropna(subset=cols_needed)

    print(f"✅ Maritime core prepared: {len(dfc)} rows, {len(predictors)} predictors.")
    return dfc, predictors

In [ ]:
maritime_predictors = [
    "Log_Origin_GDP", "Log_Destination_GDP",
    "Log_Distance", "cepii_contig", "cepii_comlang_ethno",
    "cepii_wto_o", "cepii_wto_d", "cepii_fta_wto",
    "Log_cost_per_ton_seamodes",
    "lsbci",
    "Log_origin_production_qty", "Log_destination_production_qty",
    "Log_Origin_Daily_Calorific_Food", "Log_Destination_Daily_Calorific_Food"
]

In [ ]:
def extract_coeff_table(model, r2=None, adj_r2=None, pseudo_r2=None, fe_mode="none"):
    """
    Return a clean table with:
    - parameter name
    - coefficient
    - robust SE
    - p-value
    - significance stars
    - model-level R2 / pseudo-R2
    """

    df = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "se": model.bse.values,
        "pval": model.pvalues.values
    })

    # significance stars
    def stars(p):
        if p < 0.01: return "***"
        if p < 0.05: return "**"
        if p < 0.10: return "*"
        return ""

    df["sig"] = df["pval"].apply(stars)

    # attach summary metadata
    df.attrs["r2"] = r2
    df.attrs["adj_r2"] = adj_r2
    df.attrs["pseudo_r2"] = pseudo_r2
    df.attrs["fe_mode"] = fe_mode

    return df

## Gravity Econometric Model

### OLS regression

In [ ]:
def fit_ols_maritime(
    df,
    predictors,
    fe_mode="pooled",           # "pooled", "product_fe", "od_fe"
    robust="HC3",
    eps=1e-6
):
    """
    OLS for maritime trade:
      - Dep var: log(tons_value_seamodes)
      - Predictors: logs + CEPII + maritime + food
      - FE options:
          "pooled"     : no FE
          "product_fe" : product dummies
          "od_fe"      : origin & destination ISO dummies
    Returns dict with model, X, y_log, y_level.
    """
    dfc, predictors = prepare_maritime_core(df, predictors, eps=eps)

    # --- Add fixed effects if required ---
    if fe_mode == "product_fe":
        print("\n[OLS] Adding product fixed effects...")
        prod_dum = pd.get_dummies(dfc["product_label"], prefix="PR", dtype=float)
        if prod_dum.shape[1] > 1:
            prod_dum = prod_dum.iloc[:, 1:]   # drop one to avoid dummy trap
        dfc = pd.concat([dfc, prod_dum], axis=1)
        predictors = predictors + list(prod_dum.columns)

    elif fe_mode == "od_fe":
        print("\n[OLS] Adding origin/destination fixed effects (ISO)...")
        O_dum = pd.get_dummies(dfc["origin_isocode"], prefix="O", dtype=float)
        D_dum = pd.get_dummies(dfc["destination_isocode"], prefix="D", dtype=float)
        if O_dum.shape[1] > 1:
            O_dum = O_dum.iloc[:, 1:]
        if D_dum.shape[1] > 1:
            D_dum = D_dum.iloc[:, 1:]
        dfc = pd.concat([dfc, O_dum, D_dum], axis=1)
        predictors = predictors + list(O_dum.columns) + list(D_dum.columns)

    # --- Final clean-up ---
    predictors = [p for p in predictors if p in dfc.columns]
    y_log = dfc["Log_Sea_Trade_Volume_Tons"].astype(float)
    y_level = dfc["tons_value_seamodes"].astype(float)

    X = sm.add_constant(dfc[predictors], has_constant="add").astype(float)

    print(f"\n[OLS {fe_mode}] Fitting with {len(X)} obs and {X.shape[1]} regressors...")
    model = sm.OLS(y_log, X).fit(cov_type=robust)

    print(f"[OLS {fe_mode}] Adj R² (in log-space) = {model.rsquared_adj:.3f}")
    return {
        "model": model,
        "X": X,
        "y_log": y_log,
        "y_level": y_level,
        "fe_mode": fe_mode,
        "r2": float(model.rsquared),
        "adj_r2": float(model.rsquared_adj),
        "coef_table": extract_coeff_table(model,
                                      r2=float(model.rsquared),
                                      adj_r2=float(model.rsquared_adj),
                                      fe_mode=fe_mode)
    }

In [ ]:
trade_ols_pooled      = fit_ols_maritime(df_bltr_multivar, maritime_predictors, fe_mode="pooled")
trade_ols_product_fe  = fit_ols_maritime(df_bltr_multivar, maritime_predictors, fe_mode="product_fe")

In [ ]:
trade_ols_od_fe  = fit_ols_maritime(df_bltr_multivar, maritime_predictors, fe_mode="od_fe")

In [ ]:
trade_structural_regmodel_parameter = ['const'] + maritime_predictors
trade_ols_pooled_coef_table = trade_ols_pooled["coef_table"]
trade_ols_pooled_coef_table['term_type'] = np.where(trade_ols_pooled_coef_table['term'].isin(trade_structural_regmodel_parameter),
                                                   'structural','FE_dummy')
trade_ols_pooled_coef_table.to_csv("trade_ols_pooled_coef_table.csv", index=False)


trade_ols_product_fe_coef_table = trade_ols_product_fe["coef_table"]
trade_ols_product_fe_coef_table['term_type'] = np.where(trade_ols_product_fe_coef_table['term'].isin(trade_structural_regmodel_parameter),
                                                   'structural','FE_dummy')
trade_ols_product_fe_coef_table.to_csv("trade_ols_product_fe_coef_table.csv", index=False)



trade_ols_od_fe_coef_table = trade_ols_od_fe["coef_table"]
trade_ols_od_fe_coef_table['term_type'] = np.where(trade_ols_od_fe_coef_table['term'].isin(trade_structural_regmodel_parameter),
                                                   'structural','FE_dummy')
trade_ols_od_fe_coef_table.to_csv("trade_ols_od_fe_coef_table.csv", index=False)


### PPML

In [ ]:
def fit_ppml_maritime(
    df,
    predictors,
    fe_mode="pooled",           # "pooled", "product_fe", "od_fe"
    robust="HC3",
    eps=1e-6
):
    """
    PPML gravity for maritime trade:
      - Dep var: tons_value_seamodes (levels, ≥0)
      - Predictors: logs + CEPII + maritime + food
      - FE options implemented via patsy formula.
    Returns dict with model, X, y, pseudo_r2.
    """
    dfc, predictors = prepare_maritime_core(df, predictors, eps=eps)

    # Filter out negatives; zeros are allowed
    dfc = dfc[dfc["tons_value_seamodes"] >= 0].copy()
    y = dfc["tons_value_seamodes"].astype(float)

    if fe_mode == "pooled":
        X = sm.add_constant(dfc[predictors], has_constant="add").astype(float)
        print(f"\n[PPML pooled] Fitting with {len(X)} obs and {X.shape[1]} regressors...")
        glm = sm.GLM(y, X, family=sm.families.Poisson())
        res = glm.fit(cov_type=robust)
        pr2 = pseudo_r2_mcfadden(res, y)
        print(f"[PPML pooled] McFadden pseudo R² = {pr2:.3f}")
        return {
            "model": res,
            "X": X,
            "y_level": y,
            "pseudo_r2": pr2,
            "fe_mode": fe_mode,
            "coef_table": extract_coeff_table(res,
                                      pseudo_r2=float(pr2),
                                      fe_mode=fe_mode)
        }

    elif fe_mode == "product_fe":
        print("\n[PPML product FE] Fitting with product fixed effects...")
        rhs = " + ".join(predictors + ["C(product_label)"])
        formula = f"tons_value_seamodes ~ {rhs}"
        y_mat, X = patsy.dmatrices(formula, dfc, return_type="dataframe")
        y_vec = np.asarray(y_mat).ravel()
        glm = sm.GLM(y_vec, X, family=sm.families.Poisson())
        res = glm.fit(cov_type=robust)
        pr2 = pseudo_r2_mcfadden(res, y_vec)
        print(f"[PPML product FE] McFadden pseudo R² = {pr2:.3f}")
        return {
            "model": res,
            "X": X,
            "y_level": y_vec,
            "pseudo_r2": pr2,
            "fe_mode": fe_mode,
            "coef_table": extract_coeff_table(res,
                                      pseudo_r2=float(pr2),
                                      fe_mode=fe_mode)
        }

    elif fe_mode == "od_fe":
        print("\n[PPML OD FE] Fitting with origin & destination fixed effects...")
        rhs = " + ".join(predictors + ["C(origin_isocode)", "C(destination_isocode)"])
        formula = f"tons_value_seamodes ~ {rhs}"
        y_mat, X = patsy.dmatrices(formula, dfc, return_type="dataframe")
        y_vec = np.asarray(y_mat).ravel()
        glm = sm.GLM(y_vec, X, family=sm.families.Poisson())
        res = glm.fit(cov_type=robust)
        pr2 = pseudo_r2_mcfadden(res, y_vec)
        print(f"[PPML OD FE] McFadden pseudo R² = {pr2:.3f}")
        return {
            "model": res,
            "X": X,
            "y_level": y_vec,
            "pseudo_r2": pr2,
            "fe_mode": fe_mode,
            "coef_table": extract_coeff_table(res,
                                      pseudo_r2=float(pr2),
                                      fe_mode=fe_mode)
        }

    else:
        raise ValueError("fe_mode must be 'pooled', 'product_fe', or 'od_fe'")

In [ ]:
trade_ppml_pooled     = fit_ppml_maritime(df_bltr_multivar, maritime_predictors, fe_mode="pooled")
trade_ppml_product_fe = fit_ppml_maritime(df_bltr_multivar, maritime_predictors, fe_mode="product_fe")
trade_ppml_od_fe= fit_ppml_maritime(df_bltr_multivar, maritime_predictors, fe_mode="od_fe")

In [ ]:
trade_ppml_pooled_coef_table = trade_ppml_pooled["coef_table"]
trade_ppml_pooled_coef_table['term_type'] = np.where(trade_ppml_pooled_coef_table['term'].isin(trade_structural_regmodel_parameter+['Intercept']),
                                                   'structural','FE_dummy')
trade_ppml_pooled_coef_table.to_csv("trade_ppml_pooled_coef_table.csv", index=False)

trade_ppml_product_fe_coef_table = trade_ppml_product_fe["coef_table"]
trade_ppml_product_fe_coef_table['term_type'] = np.where(trade_ppml_product_fe_coef_table['term'].isin(trade_structural_regmodel_parameter+['Intercept']),
                                                   'structural','FE_dummy')
trade_ppml_product_fe_coef_table.to_csv("trade_ppml_product_fe_coef_table.csv", index=False)

trade_ppml_od_fe_coef_table = trade_ppml_od_fe["coef_table"]
trade_ppml_od_fe_coef_table['term_type'] = np.where(trade_ppml_od_fe_coef_table['term'].isin(trade_structural_regmodel_parameter+['Intercept']),
                                                   'structural','FE_dummy')
trade_ppml_od_fe_coef_table.to_csv("trade_ppml_od_fe_coef_table.csv", index=False)

### Econometric Model evaluation

In [ ]:
def evaluate_maritime_regression(model_dict, kind):
    y_true = model_dict["y_level"]
    ln_y_true = np.log(y_true)
    X = model_dict["X"]
    if kind == "OLS":
        ln_y_pred = model_dict["model"].predict(X)
        y_pred = np.exp(ln_y_pred)
        r2 = model_dict["r2"]          # <-- OLS R² (log-space)
        adj_r2 = model_dict["adj_r2"]  # <-- Adjusted R²
    else:
        y_pred = model_dict["model"].predict(X)
        ln_y_pred = np.log(y_pred)
        r2 = model_dict.get("pseudo_r2", np.nan)   # <-- PPML pseudo R²
        adj_r2 = model_dict.get("pseudo_r2", np.nan)
    metrics = evaluate_predictions(y_true, y_pred)
    metrics["R2"] = r2
    metrics["Adj R2"] = adj_r2
    datapred = {"y_true" : y_true,
        "ln_y_true" : ln_y_true,
        "y_pred" : y_pred,
        "ln_y_pred" : ln_y_pred}
    df_datapred =pd.DataFrame(datapred)
    return {
        "metrics" : metrics,
        "datapred" : df_datapred
    }
trade_ols_pooled_metrics      = evaluate_maritime_regression(trade_ols_pooled,      kind="OLS")
trade_ols_product_fe_metrics  = evaluate_maritime_regression(trade_ols_product_fe,  kind="OLS")
trade_ols_od_fe_metrics  = evaluate_maritime_regression(trade_ols_od_fe,  kind="OLS")
trade_ppml_pooled_metrics     = evaluate_maritime_regression(trade_ppml_pooled,     kind="PPML")
trade_ppml_product_fe_metrics = evaluate_maritime_regression(trade_ppml_product_fe, kind="PPML")
trade_ppml_od_fe_metrics = evaluate_maritime_regression(trade_ppml_od_fe, kind="PPML")

In [ ]:
trade_ols_eval_tab = pd.DataFrame({
    "pooled": pd.Series(trade_ols_pooled_metrics['metrics']),
    "product fixed effect": pd.Series(trade_ols_product_fe_metrics['metrics']),
    "origin-destination fixed effect": pd.Series(trade_ols_od_fe_metrics['metrics'])
})

In [ ]:
trade_ppml_eval_tab = pd.DataFrame({
    "pooled": pd.Series(trade_ppml_pooled_metrics['metrics']),
    "product fixed effect": pd.Series(trade_ppml_product_fe_metrics['metrics']),
    "origin-destination fixed effect": pd.Series(trade_ppml_od_fe_metrics['metrics'])
})

## Machine Learning

In [ ]:
# -------------------------------
# 0. Import Required Libraries
# -------------------------------
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import shap
import matplotlib.pyplot as plt

ann_trade_features = [
    "origin_gdp_usd_current", "destination_gdp_usd_current", "cepii_dist",
    "cepii_contig", "cepii_comlang_ethno", "cepii_wto_o",
    "cepii_wto_d", "cepii_fta_wto",
    "transport_expend_per_ton_seamodes", "lsbci",
    "origin_production_qty", "destination_production_qty",
    "Origin_Daily_Calorific_Food", "Destination_Daily_Calorific_Food"
]

ann_trade_target = "tons_value_seamodes"

categorical_features = [
    "origin_isocode", "destination_isocode", "product_code"
]

target_column = "tons_value_seamodes"
target_log = "Log_Sea_Trade_Volume_Tons"

In [ ]:
numerical_features_nonlog = [
    "origin_gdp_usd_current", "destination_gdp_usd_current", "cepii_dist",
    "cepii_contig", "cepii_comlang_ethno", "cepii_wto_o", "cepii_wto_d", "cepii_fta_wto",
    "transport_expend_per_ton_seamodes", "lsbci",
    "origin_production_qty", "destination_production_qty",
    "Origin_Daily_Calorific_Food", "Destination_Daily_Calorific_Food"
]
numerical_features_nonlog
pd.DataFrame({
    "variables": numerical_features_nonlog
})

In [ ]:
# -------------------------------
# 0. Imports and Setup
# -------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# -------------------------------
# 1. Feature Settings
# -------------------------------
# numerical_features = [
#     "origin_gdp_usd_current", "destination_gdp_usd_current", "cepii_dist",
#     "cepii_contig", "cepii_comlang_ethno", "cepii_wto_o", "cepii_wto_d", "cepii_fta_wto",
#     "transport_expend_per_ton_seamodes", "lsbci",
#     "origin_production_qty", "destination_production_qty",
#     "Origin_Daily_Calorific_Food", "Destination_Daily_Calorific_Food"
# ]

# numerical_features = [
#     "Log_Origin_GDP", "Log_Destination_GDP", "Log_Distance",
#     "cepii_contig", "cepii_comlang_ethno", "cepii_wto_o", "cepii_wto_d", "cepii_fta_wto",
#     "Log_cost_per_ton_seamodes", "lsbci",
#     "Log_origin_production_qty", "Log_destination_production_qty",
#     "Log_Origin_Daily_Calorific_Food", "Log_Destination_Daily_Calorific_Food"
# ]


numerical_features = [
    "Log_Origin_GDP", "Log_Destination_GDP",
    "Log_cost_per_ton_seamodes", "lsbci", 
    "cepii_contig", "cepii_comlang_ethno", "cepii_wto_o", "cepii_wto_d", "cepii_fta_wto",
    "Log_origin_production_qty", "Log_destination_production_qty",
    "Log_Origin_Daily_Calorific_Food", "Log_Destination_Daily_Calorific_Food"
]

categorical_features = ["country_pair_id", "product_code"]

target_column = "tons_value_seamodes"
target_log = "log_tons_value_seamodes"
all_features = numerical_features + categorical_features

# -------------------------------
# 2. Utility Functions
# -------------------------------
def safe_log(x):
    return np.log(np.where(x > 0, x, 1e-9))

def safe_exp(x):
 ##   return np.where(np.exp(x)<0,0,np.exp(x))
    return np.exp(x)

def bias_corrected_exp(log_pred, log_residuals):
    sigma2 = np.var(log_residuals)
    return np.exp(log_pred + 0.5 * sigma2),sigma2 

def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    extreme = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true)) > 0.3) * 100
    return r2, rmse, mae, mape, extreme

**Publication review note.** The active split is random 80/20 within each year/product stratum, followed by a 0.1-ton threshold. TimeSeriesSplit operates on the resulting row order, not explicit held-out years. The original split is preserved.


In [ ]:
# ----------------------------------
# 3. Prepare Cleaned Dataset
# ----------------------------------
df_all = df_bltr_multivar.copy()
df_all["year"] = df_all["year"].astype(int)
df_all[target_log] = safe_log(df_all[target_column])
df_all[target_column] = np.where(df_all[target_column].isna(),0,df_all[target_column])
df_all["country_pair_id"] = df_all["origin_isocode"] + "_" + df_all["destination_isocode"]

# Drop rows with missing required values
required_cols = all_features + [target_log]
df_all = df_all.dropna(subset=required_cols)

# Train-test split
# train_years = [2016, 2017, 2018, 2019,2020]
# test_years = [2021]

from sklearn.model_selection import train_test_split

def stratified_time_window_split(df, year_col="year", stratify_col="product_code", train_ratio=0.8, random_state=42):
    """
    For each year and product_code combination, split into train/test (e.g., 80/20)
    Returns two DataFrames: df_train, df_test
    """
    train_parts = []
    test_parts = []
    
    grouped = df.groupby([year_col, stratify_col])
    
    for (year, product), group in grouped:
        if len(group) < 2:
            # Not enough rows to split, keep in train
            train_parts.append(group)
        else:
            train, test = train_test_split(group, train_size=train_ratio, random_state=random_state)
            train_parts.append(train)
            test_parts.append(test)
    
    df_train = pd.concat(train_parts).reset_index(drop=True)
    df_test = pd.concat(test_parts).reset_index(drop=True)
    
    return df_train, df_test

df_train, df_test = stratified_time_window_split(df_all, year_col="year", stratify_col="product_code", train_ratio=0.8)
X_all = df_all[numerical_features + categorical_features]
Y_all = df_all[target_log]
trade_threshold = 0.1
df_train = df_train[df_train["tons_value_seamodes"] >= trade_threshold].copy()
df_test = df_test[df_test["tons_value_seamodes"] >= trade_threshold].copy()
X_train = df_train[numerical_features + categorical_features]
y_train = df_train[target_log]
X_test  = df_test[numerical_features + categorical_features]
y_test  = df_test[target_log]

# ----------------------------------
# 4. Preprocessing Pipeline
# ----------------------------------
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

# ----------------------------------
# 5. Build ANN Pipeline
# ----------------------------------
ann_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("ann", MLPRegressor(
        activation="relu",
        solver="adam",
        early_stopping=True,
        validation_fraction=0.1,
        max_iter=500,
        random_state=42
    ))
])

# Grid search parameters
param_grid = {
    "ann__hidden_layer_sizes": [ (64,) * 10,        # 10 hidden layers with 64 neurons each
                               ],
    "ann__alpha": [0.0001, 0.001],
    "ann__learning_rate_init": [0.001]
}

tscv = TimeSeriesSplit(n_splits=3)
grid_ann = GridSearchCV(ann_pipeline, param_grid, cv=tscv, scoring="r2", verbose=1, n_jobs=-1)
grid_ann.fit(X_train, y_train)

# Save model
joblib.dump(grid_ann.best_estimator_, "trade_ann_log_model.pkl")

print("✅ Best ANN Params:", grid_ann.best_params_)

In [ ]:
# For reporting
report_datatrainsplit = (
    df_train.groupby(['year','product_label'])
    .agg({'country_pair_id' : 'count'})
    .round(3)
).reset_index().rename(columns={"country_pair_id": "Train_count"})
report_datatestsplit = (
    df_test.groupby(['year','product_label'])
    .agg({'country_pair_id' : 'count'})
    .round(3)
).reset_index().rename(columns={"country_pair_id": "Test_count"})
report_data = report_datatrainsplit.merge(
        report_datatestsplit[['year','product_label','Test_count']],
        on=['year','product_label'],
        how="left"
    )
report_data

In [ ]:
import matplotlib.pyplot as plt
import shap
import numpy as np

# ----------------------------------
# 6. Evaluation Function
# ----------------------------------
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    extreme_share = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true)) > 0.3) * 100
    return r2, rmse, mae, mape, extreme_share

# ----------------------------------
# 7. Predict and Evaluate on Test Set (log scale)
# ----------------------------------
y_pred_log = grid_ann.predict(X_test)
log_residuals = y_test - y_pred_log
r2, rmse, mae, mape, extreme = compute_metrics(y_test, y_pred_log)

print("\n📊 ANN Performance (Log-Scale, Test Set)")
print(f"R²     : {r2:.4f}")
print(f"RMSE   : {rmse:.4f}")
print(f"MAE    : {mae:.4f}")
print(f"MAPE   : {mape:.2f}%")
print(f"Extreme Error >30%: {extreme:.2f}%")

# ----------------------------------
# 8. Predict and Evaluate (Level)
# ----------------------------------
y_pred_level = safe_exp(y_pred_log)
y_pred_level_corrected,sigma2 = bias_corrected_exp(y_pred_log, log_residuals)
y_true_level = safe_exp(y_test)
r2_level, rmse_level, mae_level, mape_level, extreme_level = compute_metrics(y_true_level, y_pred_level)

print("\n📊 ANN Performance (Level, Test Set)")
print(f"R²     : {r2_level:.4f}")
print(f"RMSE   : {rmse_level:,.2f}")
print(f"MAE    : {mae_level:,.2f}")
print(f"MAPE   : {mape_level:.2f}%")
print(f"Extreme Error >30%: {extreme_level:.2f}%")

# ----------------------------------
# 9. Export all data Predictions
# ----------------------------------
y_pred_log = grid_ann.predict(X_all)
log_residuals_all = Y_all - y_pred_log
y_pred_level_corr,sigma2 = bias_corrected_exp(y_pred_log, log_residuals_all)
pred_df = X_all.copy()
pred_df["year"] = df_all["year"].values
pred_df["y_true_log"] = Y_all
pred_df["y_pred_log"] = y_pred_log
pred_df["y_log_residuals"] = log_residuals_all
pred_df["y_sigma2"] = sigma2
pred_df["y_true"] = df_all[target_column]
pred_df["y_pred"] = safe_exp(y_pred_log)
pred_df.to_csv("trade_ann_predictions_log_based.csv", index=False)
print("✅ Predictions saved to trade_ann_predictions_log_based.csv")

# ----------------------------------
# 10. Plot Loss Curve
# ----------------------------------
plt.plot(grid_ann.best_estimator_.named_steps["ann"].loss_curve_)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve – Trade ANN (Log Y)")
plt.grid(True)
plt.tight_layout()
plt.savefig("trade_ann_log_loss_curve.png", dpi=300)
print("✅ Loss curve saved.")

# ----------------------------------
# 11. SHAP Analysis
# ----------------------------------
print("🧠 Running SHAP analysis on Trade ANN...")

X_trans = grid_ann.best_estimator_.named_steps["preprocess"].transform(X_test)

explainer = shap.Explainer(
    grid_ann.best_estimator_.named_steps["ann"].predict,
    X_trans[:50],  # background
    feature_names=grid_ann.best_estimator_.named_steps["preprocess"].get_feature_names_out()
)

shap_values = explainer(X_trans[:50])
shap.summary_plot(shap_values, X_trans[:50], show=False)
plt.title("SHAP Summary – Trade ANN (log scale)")
plt.tight_layout()
plt.savefig("shap_trade_ann_log_summary.png", dpi=300)
print("✅ SHAP summary plot saved.")

In [ ]:
trade_feature_names = grid_ann.best_estimator_.named_steps["preprocess"].get_feature_names_out()
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# 1. Compute mean absolute SHAP values per feature
# shap_values shape: (n_samples, n_features)
# ------------------------------------------------------------
mean_abs_shap = np.mean(np.abs(shap_values.values), axis=0)

# ------------------------------------------------------------
# 2. Parse feature type and variable name
# ------------------------------------------------------------
records = []

for i, fname in enumerate(trade_feature_names):
    
    # Default classification
    feature_type = "numerical"
    variable = fname
    
    # --- Identify categorical groups ---
    if "product_code" in fname:
        feature_type = "product_code"
        variable = re.sub(r"cat__product_code_", "", fname)

    elif "country_pair_id" in fname:
        feature_type = "country_pair_id"
        variable = re.sub(r"cat__country_pair_id_", "", fname)

    elif fname.startswith("num__"):
        feature_type = "numerical"
        variable = fname.replace("num__", "")

    # Save row
    records.append({
        "rank": None,                  # placeholder, fill later
        "feature_category": feature_type,
        "variable": variable,
        "mean_abs_SHAP": mean_abs_shap[i]
    })

# ------------------------------------------------------------
# 3. Build DataFrame and sort by SHAP
# ------------------------------------------------------------
shap_df = pd.DataFrame(records)
shap_df = shap_df.sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)

# Add ranking column
shap_df["rank"] = shap_df.index + 1

# Reorder columns
shap_df = shap_df[["rank", "feature_category", "variable", "mean_abs_SHAP"]]

# ------------------------------------------------------------
# 4. Display and export
# ------------------------------------------------------------

shap_df.to_csv("trade_ann_shap_importance_table.csv", index=False)

print("✅ SHAP Importance Table saved: trade_ann_shap_importance_table.csv")

In [ ]:
pred_df['y_log_residual'] = pred_df['y_true_log']-pred_df['y_pred_log']
pred_df['y_residual'] = pred_df['y_true']-pred_df['y_pred']
pred_df['y_perc_error']= (np.abs((pred_df['y_true'] - pred_df['y_pred']) / np.where(pred_df['y_true'] == 0, 1e-9, pred_df['y_true'])) * 100)

In [ ]:
def evaluate_ml_model(y_true, y_pred, label="Model"):
    y_df = pd.concat([y_true,pd.Series(y_pred,index=y_true.index, name = 'y_pred')],axis=1, ignore_index = True)
    y_pred_s = y_df[1].loc[y_df[y_df[0]>0.1].index]
    y_true_s = y_df[0].loc[y_df[y_df[0]>0.1].index]
    r2 = r2_score(y_true_s, y_pred_s)
    rmse = np.sqrt(mean_squared_error(y_true_s, y_pred_s))
    mae = mean_absolute_error(y_true_s, y_pred_s)
    mape_adj = np.mean(np.abs((y_true_s - y_pred_s) / np.where(y_true_s == 0, 1e-9, y_true_s))) * 100
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100

    # Extreme error analysis
    abs_pct_error = np.abs((y_true_s - y_pred_s) / np.where(y_true_s == 0, 1e-9, y_true_s))
    extreme_mask = abs_pct_error > 0.3  # >30%
    extreme_count = extreme_mask.sum()
    total = len(y_true)
    extreme_share = extreme_count / total * 100

    return {
        "Model": label,
        "R2": round(r2, 4),
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "MAPE (%)": round(mape_adj, 2),
        "Extreme Errors (>30%)": int(extreme_count),
        "Extreme Share (%)": round(extreme_share, 2)
    }
y_tann = pred_df["y_true_log"]
y_pred_tann = pred_df['y_pred_log']
metrics_tann = evaluate_ml_model(y_tann, y_pred_tann, label="ANN")


In [ ]:
y


# Dashboard export — Transport Cost Models

Packages existing results only. Ridge is held-out; ANN is in-sample on all data. The implemented target is natural-log transport cost per ton (`ln_cost_tonkm` internally). Run only with the existing transport-result objects available; missing objects stop export without fitting models.


In [ ]:
def _dashboard_export_transport_cost_current_results():
    # Local names protect all existing notebook variables and model objects.
    from pathlib import Path
    import numpy as np
    import pandas as pd

    required = ('pooled_metrics', 'results', 'all_diag', 'metrics_ridge',
                'ridge_pred_df', 'metrics_ann', 'ann_pred_df', 'df_ann_model')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Export stopped; existing notebook objects unavailable: '
                           + ', '.join(missing) + '. No modelling cells were rerun.')

    rename = {'R2': 'r2', 'RMSE': 'rmse', 'MAE': 'mae', 'MAPE(%)': 'mape',
              'MAPE (%)': 'mape', 'Bias': 'bias', 'ExtremeResiduals': 'extreme_residuals',
              'ExtremeShare(%)': 'extreme_share', 'Extreme Share (%)': 'extreme_share',
              'Extreme Errors (>30%)': 'extreme_errors', 'N_obs': 'n_obs'}
    metric_cols = ['r2', 'rmse', 'mae', 'mape', 'bias',
                   'extreme_residuals', 'extreme_share', 'n_obs']
    pooled = pooled_metrics.copy().rename(columns=rename)
    pooled.insert(0, 'model', pooled.index)
    pooled = pooled.reindex(columns=['model'] + metric_cols)
    assert set(pooled['model']).issubset({'OLS', 'GLM'}), 'Unexpected pooled model labels'

    coefficient_parts = []
    for label, key in [('OLS', 'ols_params'), ('GLM', 'gamma_params')]:
        if key in results:
            part = results[key].copy().rename(columns={
                'coef': 'coefficient', 'se': 'std_error', 'p': 'p_value'})
            part.insert(0, 'variable', part.index)
            part.insert(0, 'model', label)
            coefficient_parts.append(part[['model', 'variable', 'coefficient',
                                           'std_error', 'p_value']])
    assert coefficient_parts, 'No stored pooled coefficients'
    coefficients = pd.concat(coefficient_parts, ignore_index=True)

    fe = all_diag.copy().rename(columns=rename).reindex(columns=['commodity'] + metric_cols)
    # n_obs is not stored in all_diag; retain NaN instead of guessing or joining.

    ann = ann_pred_df.copy()
    source = df_ann_model
    assert ann.index.is_unique and source.index.is_unique, 'ANN source indices must be unique'
    assert ann.index.equals(source.index), 'ANN/source row order differs'
    assert len(ann['y_true']) == len(ann['y_pred']) == len(source), 'ANN lengths differ'
    pd.testing.assert_series_equal(ann['y_true'], source['ln_cost_tonkm'], check_names=False)
    identifiers = ['year', 'origin_isocode', 'destination_isocode', 'product_code']
    for column in identifiers:
        pd.testing.assert_series_equal(ann[column], source[column], check_names=False)
    assert np.allclose(source['ln_cost_tonkm'],
                       np.log(np.clip(source['cost_tonkm_sea'].astype(float), 1e-9, None)),
                       equal_nan=True), 'Unexpected ANN target transformation'
    predictions = ann[identifiers + ['y_true', 'y_pred']].copy().rename(columns={
        'y_true': 'actual_log_cost_per_ton', 'y_pred': 'predicted_log_cost_per_ton'})
    omitted = []
    # Labels are optional: use exact source indices plus value/ID checks, never a join.
    raw = globals().get('df_bltr_multivar')
    raw_aligned = None
    if isinstance(raw, pd.DataFrame) and raw.index.is_unique and ann.index.isin(raw.index).all():
        candidate = raw.loc[ann.index]
        ids_match = all(candidate[c].equals(source[c]) for c in identifiers)
        target_matches = ('transport_expend_per_ton_seamodes' in candidate and
                          candidate['transport_expend_per_ton_seamodes'].equals(source['cost_tonkm_sea']))
        if ids_match and target_matches:
            raw_aligned = candidate
    for column in ['origin_label', 'destination_label', 'product_label']:
        if raw_aligned is not None and column in raw_aligned:
            predictions[column] = raw_aligned[column]
        else:
            omitted.append(column)
    predictions['evaluation_scope'] = 'in_sample_all_data'
    predictions['source_target_variable'] = 'ln_cost_tonkm'
    order = ['year', 'origin_label', 'destination_label', 'product_label',
             'origin_isocode', 'destination_isocode', 'product_code',
             'actual_log_cost_per_ton', 'predicted_log_cost_per_ton',
             'evaluation_scope', 'source_target_variable']
    predictions = predictions[[c for c in order if c in predictions]]

    def period(values):
        return ', '.join(str(v) for v in sorted(values.dropna().unique()))

    ridge_period = np.nan
    ridge_source = globals().get('df_ml_model')
    if isinstance(ridge_source, pd.DataFrame) and ridge_source.index.is_unique and ridge_pred_df.index.is_unique:
        if ridge_pred_df.index.isin(ridge_source.index).all():
            candidate = ridge_source.loc[ridge_pred_df.index]
            columns = [c for c in ridge_pred_df if c not in ('y_true', 'y_pred')]
            if (all(ridge_pred_df[c].equals(candidate[c]) for c in columns)
                    and ridge_pred_df['y_true'].equals(candidate['ln_cost_tonkm'])):
                ridge_period = period(candidate['year'])
    rows = []
    for label, metrics, frame, scope, years in [
        ('Ridge Regression', metrics_ridge, ridge_pred_df, 'held_out_test', ridge_period),
        ('ANN', metrics_ann, ann, 'in_sample_all_data', period(ann['year']))
    ]:
        assert len(frame['y_true']) == len(frame['y_pred'])
        row = {rename.get(k, k): v for k, v in metrics.items()}
        row.update(model=label, n_obs=len(frame), evaluation_scope=scope,
                   evaluation_period=years, target='log_transport_cost_per_ton',
                   source_target_variable='ln_cost_tonkm')
        rows.append(row)
    ml_cols = ['model', 'r2', 'rmse', 'mae', 'mape', 'bias', 'extreme_errors',
               'extreme_share', 'n_obs', 'evaluation_scope', 'evaluation_period',
               'target', 'source_target_variable']
    ml = pd.DataFrame(rows).reindex(columns=ml_cols)
    tables = {'transport_cost_pooled_metrics.csv': (pooled, ['model']),
              'transport_cost_pooled_coefficients.csv': (coefficients, ['model', 'variable']),
              'transport_cost_fe_metrics.csv': (fe, ['commodity']),
              'transport_cost_ml_metrics.csv': (ml, ['model']),
              'transport_cost_ann_predictions.csv': (predictions, identifiers)}
    numeric = set(metric_cols + ['extreme_errors', 'coefficient', 'std_error', 'p_value',
                                'actual_log_cost_per_ton', 'predicted_log_cost_per_ton', 'year'])
    # Validate every table before writing any file. No rows are dropped or values filled.
    for filename, (frame, identity) in tables.items():
        assert not frame.empty, filename + ': empty table'
        assert frame.columns.is_unique, filename + ': duplicate columns'
        assert not frame[identity].isna().any().any(), filename + ': missing identity'
        duplicates = int(frame.duplicated(identity).sum())
        assert duplicates == 0, filename + ': duplicate identities'
        for column in numeric.intersection(frame.columns):
            pd.to_numeric(frame[column], errors='raise')
        print(filename, '\n rows:', len(frame), '\n columns:', frame.columns.tolist(),
              '\n duplicate identities:', duplicates, '\n missing:', frame.isna().sum().to_dict())
    print('ANN rows:', len(predictions), '; min/max year:', ann['year'].min(), ann['year'].max(),
          '; available years:', period(ann['year']), '; commodities:', ann['product_code'].unique().tolist(),
          '; actual/prediction lengths equal:', len(ann['y_true']) == len(ann['y_pred']),
          '; identifier alignment: exact unique source index and matching identifier/target values',
          '; missing actual:', ann['y_true'].isna().sum(), '; missing predicted:', ann['y_pred'].isna().sum(),
          '; evaluation_scope: in_sample_all_data; target: natural log of USD per ton',
          '; omitted labels:', omitted)
    print('Pooled models:', pooled['model'].tolist(), '; FE commodities:', fe['commodity'].tolist())
    print('WARNING:\nRidge evaluation scope = held_out_test\nANN evaluation scope = in_sample_all_data\n'
          'These metrics are not directly comparable as equivalent out-of-sample evaluations.')
    print('Metric values and percentage units are preserved; unavailable fields remain NaN. '
          'Pooled OLS and GLM use different target scales; FE metrics mix log R2 and level errors. '
          'Existing extreme-error definitions are unchanged.')
    output = Path("dashboard_exports")
    output.mkdir(parents=True, exist_ok=True)
    for filename, (frame, _) in tables.items():
        frame.to_csv(output / filename, index=False)
        print('Saved:', output / filename, 'rows:', len(frame))

_dashboard_export_transport_cost_current_results()
